# PlugIR · Fresh-sample caption replication

Run vanilla, guaranteed target caption, and random non-target caption on the same 200 previously unused dialogues. Each dialogue stops at the first top-10 hit or after two questions. The main comparison is cumulative Hits@10 after one and two questions. Two workers run concurrently, and the third arm starts when a worker frees up.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# Start Jupyter in the directory containing this notebook, or set PACKAGE explicitly.
PACKAGE = Path.cwd().resolve()
RESULTS = PACKAGE / "results"
REPORT = PACKAGE / "source" / "report_hits.py"

# Change these paths if your checkout or MLX environment lives elsewhere.
PROJECT = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / "ChatIR-Cornell2026").is_dir()),
    PACKAGE.parent,
)
REPO = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / ".git").exists()),
    PROJECT / "ChatIR-Cornell2026",
)
BLIP_PYTHON = REPO / ".venv" / "bin" / "python"
MLX_ENV = PROJECT / "tmp" / "agentir-mlx-vlm-env"
QWEN_REVISION = "8b2b98c00a6b4d291155e4890773ca8f769aee53"
SNAPSHOT = (REPO / "cache/huggingface/hub/models--mlx-community--Qwen3.5-9B-4bit"
            / "snapshots" / QWEN_REVISION)
ARMS = ["vanilla", "target_caption", "random_caption_swap"]
SAMPLE = json.loads((PACKAGE / "sample.json").read_text())
DIALOGUE_COUNT = len(SAMPLE["tasks"])
print(f"{DIALOGUE_COUNT} fresh dialogues per method, seed {SAMPLE['seed']}")

## Check saved progress

In [ ]:
def saved_ids(arm):
    return {int(file.stem) for file in (RESULTS / arm / arm / "dialogues").glob("*.json")}

for arm in ARMS:
    print(f"{arm}: {len(saved_ids(arm))}/{DIALOGUE_COUNT}")
print(f"Matched: {len(set.intersection(*(saved_ids(arm) for arm in ARMS)))}/{DIALOGUE_COUNT}")

## Run or resume on Apple Silicon

Use this cell to start or resume the three runs from the saved checkpoints. While a run is active, use the progress cell.

In [ ]:
subprocess.run(
    ["caffeinate", "-i", str(BLIP_PYTHON), "-u", str(PACKAGE / "source/launch.py"),
     "--snapshot", str(SNAPSHOT), "--output", str(RESULTS), "--report-python", sys.executable],
    check=True,
)

## Plot the matched comparison

The default requires all 200 dialogues for all three methods.

In [ ]:
from IPython.display import Image, display

ALLOW_PARTIAL = False
command = [sys.executable, str(REPORT), str(RESULTS)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")
figures = subprocess.check_output(command, text=True).strip()
display(Image(filename=str(Path(figures) / "hits_at_10.png")))